My final model is as follows

- re training the fourth layer and the final layer
- final layer lr=0.01
- fourth layer lr=0.0001
- using adam optimiser
- 3 epochs
- batch size 32
- pre trained model from torchgeo.models import ResNet18_Weights, resnet18, weights = ResNet18_Weights.SENTINEL2_RGB_MOCO
- my final layer needs 10 output classes
- my images are the eurostat 64x64 images
- freeze all other layers
- track memory and time for each epoch
- test model on test set

In [ ]:
import torch
import torch.nn as nn
from torchgeo.models import ResNet18_Weights, resnet18
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
import timm

In [ ]:
# Step 1: Load training images temporarily without normalisation
initial_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

initial_train_dataset = datasets.ImageFolder(
    root="../../data/train",
    transform=initial_transform
)

# Step 2: Calculate RGB mean and standard deviation
stats_loader = DataLoader(
    initial_train_dataset,
    batch_size=32,
    shuffle=False
)

channel_sum = torch.zeros(3)
channel_squared_sum = torch.zeros(3)
total_pixels = 0

for images, labels in stats_loader:
    channel_sum += images.sum(dim=[0, 2, 3])
    channel_squared_sum += (images ** 2).sum(dim=[0, 2, 3])
    total_pixels += images.shape[0] * images.shape[2] * images.shape[3]

mean = channel_sum / total_pixels
std = torch.sqrt(channel_squared_sum / total_pixels - mean ** 2)

print("RGB mean:", mean.tolist())
print("RGB standard deviation:", std.tolist())

# Step 3: Create preprocessing transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])

# Step 4: Load the final training and test datasets
train_dataset = datasets.ImageFolder(
    root="../../data/train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="../../data/test",
    transform=transform
)

# Step 5: Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# Step 6: Check the datasets
print("\nClasses:", train_dataset.classes)
print("Number of classes:", len(train_dataset.classes))
print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))

# Check a batch of images
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)